In [ ]:
import gradio as gr
import cv2
import numpy as np
from PIL import Image
import tensorflow as tf
from sklearn.preprocessing import LabelEncoder
import json
import os
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input

# === Constants ===
IMG_SIZE = 224  # Input size for the model

# === Load model ===
model_path = "MobileNetV2_224size_64bs_15E_all_trainable_255prepro.h5"
model = tf.keras.models.load_model(model_path)

# === Load COCO metadata ===
base_path = 'nutritionverse-data'
coco_json_path = os.path.join(base_path, 'nutritionverse-manual/nutritionverse-manual/images/_annotations.coco.json')

with open(coco_json_path, 'r') as f:
    coco_data = json.load(f)

# === Label encoder ===
category_ids = [cat['id'] for cat in coco_data['categories']]
label_encoder = LabelEncoder()
label_encoder.fit(category_ids)

def detect_food_in_image(image):
    image = np.array(image)
    image = cv2.cvtColor(image, cv2.COLOR_RGB2BGR)
    if image is None:
        raise ValueError("Invalid image input.")

    image = cv2.cvtColor(image, cv2.COLOR_BGR2RGB)
    original_image = image.copy()
    results = []

    window_sizes = [
        (image.shape[1] // 2, image.shape[0] // 2),
        (image.shape[1] // 3, image.shape[0] // 3)
    ]
    step_divisors = [2, 4, 8, 12]
    sliding_window_results = []

    for win_w, win_h in window_sizes:
        if win_w < 50 or win_h < 50:
            continue
        for divisor in step_divisors:
            step_size = max(1, min(image.shape[0], image.shape[1]) // divisor)
            for y in range(0, image.shape[0] - win_h, step_size):
                for x in range(0, image.shape[1] - win_w, step_size):
                    window = original_image[y:y+win_h, x:x+win_w]
                    resized = cv2.resize(window, (IMG_SIZE, IMG_SIZE))
                    input_img = preprocess_input(resized.astype(np.float32))
                    prediction = model.predict(np.expand_dims(input_img, axis=0), verbose=0)
                    predicted_class_idx = np.argmax(prediction)
                    confidence = prediction[0][predicted_class_idx]

                    if confidence < 0.85:
                        continue

                    predicted_class = label_encoder.inverse_transform([predicted_class_idx])[0]
                    category_name = next((cat['name'] for cat in coco_data['categories'] if cat['id'] == predicted_class), "Unknown")

                    sliding_window_results.append({
                        'class_idx': predicted_class_idx,
                        'category': category_name,
                        'confidence': float(confidence),
                        'bbox': [x, y, win_w, win_h]
                    })

    # Non-Maximum Suppression (NMS)
    sliding_window_results.sort(key=lambda x: x['confidence'], reverse=True)
    keep_results = []

    for current in sliding_window_results:
        x1, y1, w1, h1 = current['bbox']
        x2, y2 = x1 + w1, y1 + h1
        overlaps = False

        for kept in keep_results:
            x3, y3, w2, h2 = kept['bbox']
            x4, y4 = x3 + w2, y3 + h2
            inter_x = max(0, min(x2, x4) - max(x1, x3))
            inter_y = max(0, min(y2, y4) - max(y1, y3))
            intersection = inter_x * inter_y
            union = w1*h1 + w2*h2 - intersection
            iou = intersection / union if union > 0 else 0
            if iou > 0.1 and current['category'] == kept['category']:
                overlaps = True
                break
        if not overlaps:
            keep_results.append(current)

    # Draw results
    for result in keep_results:
        x, y, w, h = result['bbox']
        category_name = result['category']
        confidence = result['confidence']
        cv2.rectangle(original_image, (x, y), (x+w, y+h), (0, 0, 255), 2)
        cv2.putText(original_image, f"{category_name} ({confidence:.2f})",
                    (x, y - 10), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255, 255, 255), 2)
        results.append(result)

    boxed_image = Image.fromarray(original_image)
    return boxed_image, results

# === Gradio Interface ===
interface = gr.Interface(
    fn=detect_food_in_image,
    inputs=gr.Image(type="pil"),
    outputs=[
        gr.Image(label="Detected Image"),
        gr.JSON(label="Detection Results")
    ],
    title="Food Detection with Sliding Window",
    description="Upload an image to detect food using a custom trained MobileNetV2 model with a sliding window strategy."
)

# === Launch ===
interface.launch()


* Running on local URL:  http://127.0.0.1:7868
* To create a public link, set `share=True` in `launch()`.


In [4]:
import gradio as gr
import tensorflow as tf
import numpy as np
import cv2
import os
import json
from PIL import Image, ImageDraw, ImageFont
from sklearn.preprocessing import LabelEncoder
import matplotlib.pyplot as plt
from PIL import Image

# Define constants
IMG_SIZE = 224  # Input size for the model

# Load the model
model_path = "MobileNetV2_224size_64bs_15E_all_trainable_255prepro.h5"  # Change to your .h5 model path if needed
model = tf.keras.models.load_model(model_path)

# Setting paths
base_path = 'nutritionverse-data'
coco_json_path = os.path.join(base_path, 'nutritionverse-manual/nutritionverse-manual/images/_annotations.coco.json')

# Load COCO format metadata
with open(coco_json_path, 'r') as f:
    coco_data = json.load(f)

# Create LabelEncoder
category_ids = [cat['id'] for cat in coco_data['categories']]
label_encoder = LabelEncoder()
label_encoder.fit(category_ids)

IMG_SIZE = 224  # Match your model's input size

def detect_food_in_image_pil(image: Image.Image) -> tuple:
    image = np.array(image.convert("RGB"))
    original_image = image.copy()
    results = []

     # Define window sizes (relative to image size)
    window_sizes = [
        (image.shape[1]//2, image.shape[0]//2),  # Half image size
        (image.shape[1]//3, image.shape[0]//3),  # One-third size
        # (image.shape[1]//4, image.shape[0]//4)   # One-fourth size
    ]
    
    # Smaller step size to catch more food items (with overlap)
    step_size = min(image.shape[0], image.shape[1]) // 10

    sliding_window_results = []

    for win_w, win_h in window_sizes:
        if win_w < 50 or win_h < 50:
            continue

        for y in range(0, image.shape[0] - win_h, step_size):
            for x in range(0, image.shape[1] - win_w, step_size):
                window = original_image[y:y + win_h, x:x + win_w]
                window_resized = cv2.resize(window, (IMG_SIZE, IMG_SIZE))
                window_normalized = window_resized / 255.0
                prediction = model.predict(np.expand_dims(window_normalized, axis=0), verbose=0)
                predicted_class_idx = np.argmax(prediction)
                confidence = prediction[0][predicted_class_idx]

                if confidence < 0.5:
                    continue

                predicted_class = label_encoder.inverse_transform([predicted_class_idx])[0]
                category_name = next((cat['name'] for cat in coco_data['categories']
                                      if cat['id'] == predicted_class), "Unknown")

                sliding_window_results.append({
                    'class_idx': predicted_class_idx,
                    'category': category_name,
                    'confidence': float(confidence),
                    'bbox': [x, y, win_w, win_h]
                })

    # NMS
    sliding_window_results.sort(key=lambda x: x['confidence'], reverse=True)
    keep_results = []

    # Non-maximum suppression to eliminate overlapping windows
    # Sort sliding window results by confidence (highest first)
    sliding_window_results.sort(key=lambda x: x['confidence'], reverse=True)
    
    # Keep track of which results to keep
    keep_results = []
    
    # Simple NMS implementation
    for i, current in enumerate(sliding_window_results):
        # Skip if this window was already suppressed
        if current in keep_results:
            continue
            
        # Current window coordinates
        x1, y1, w1, h1 = current['bbox']
        x2, y2 = x1 + w1, y1 + h1
        
        # Keep this window
        keep_results.append(current)
        
        # Check against all other windows
        for j, other in enumerate(sliding_window_results):
            if i == j or other in keep_results:
                continue
                
            # Other window coordinates
            x3, y3, w2, h2 = other['bbox']
            x4, y4 = x3 + w2, y3 + h2
            
            # Calculate intersection area
            x_overlap = max(0, min(x2, x4) - max(x1, x3))
            y_overlap = max(0, min(y2, y4) - max(y1, y3))
            intersection = x_overlap * y_overlap
            
            # Calculate union area
            area1 = w1 * h1
            area2 = w2 * h2
            union = area1 + area2 - intersection
            
            # Calculate IoU (Intersection over Union)
            iou = intersection / union if union > 0 else 0
            
            # If there's significant overlap and same category, suppress the lower confidence window
            if iou > 0.5 and current['category'] == other['category']:
                # Don't add other to keep_results - it's being suppressed
                pass
    
    # Add the filtered sliding window results to the overall results
    for result in keep_results:
        x, y, w, h = result['bbox']
        category_name = result['category']
        confidence = result['confidence']
        
        # Add to results if not already detected by contour method
        # (Simple check - if there's no major overlap with existing results)
        should_add = True
        for existing in results:
            ex, ey, ew, eh = existing['bbox']
            if (x < ex + ew and x + w > ex and 
                y < ey + eh and y + h > ey and
                existing['category'] == category_name):
                should_add = False
                break
                
        if should_add:
            results.append(result)
            
            # Draw rectangle and label on the image
            rect = plt.Rectangle((x, y), w, h, linewidth=2, edgecolor='b', facecolor='none')
            plt.gca().add_patch(rect)
            plt.text(x, y - 5, f"{category_name} ({confidence:.2f})", 
                     color='white', fontsize=12, backgroundcolor='blue')
    
    # Display results
    plt.axis('off')
    plt.show()

    for i, current in enumerate(sliding_window_results):
        x1, y1, w1, h1 = current['bbox']
        x2, y2 = x1 + w1, y1 + h1
        keep_results.append(current)

        for j, other in enumerate(sliding_window_results):
            if i == j or other in keep_results:
                continue
            x3, y3, w2, h2 = other['bbox']
            x4, y4 = x3 + w2, y3 + h2
            x_overlap = max(0, min(x2, x4) - max(x1, x3))
            y_overlap = max(0, min(y2, y4) - max(y1, y3))
            intersection = x_overlap * y_overlap
            area1 = w1 * h1
            area2 = w2 * h2
            union = area1 + area2 - intersection
            iou = intersection / union if union > 0 else 0
            if iou > 0.9 and current['category'] == other['category']:
                continue

    # Draw on image using PIL
    annotated_image = Image.fromarray(original_image)
    draw = ImageDraw.Draw(annotated_image)
    font = ImageFont.load_default()

    for result in keep_results:
        x, y, w, h = result['bbox']
        # draw.rectangle([(x, y), (x + w, y + h)], outline="blue", width=3)
        # draw.text((x, y - 10), f"{result['category']} ({result['confidence']:.2f})", fill="blue", font=font)
        results.append(result)

    return annotated_image, results


# Gradio Interface
interface = gr.Interface(
    fn=detect_food_in_image_pil,
    inputs=gr.Image(type="pil"),
    outputs=[
        gr.Image(label="Annotated Image"),
        gr.JSON(label="Detection Results")
    ],
    title="Food Detection",
    description="Upload an image to detect foods using a custom trained CNN model."
)

interface.launch()

* Running on local URL:  http://127.0.0.1:7863
* To create a public link, set `share=True` in `launch()`.


C:\Users\RAYMOND\AppData\Local\Temp\ipykernel_16604\3194285632.py:157: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
